# Feature Engineering — Home Credit Default Risk

This notebook transforms the raw `application_train.csv` file into a modeling-ready dataset: dropping sparse columns, converting Home Credit's `DAYS_*` fields into interpretable ages/tenures, engineering ratio and aggregate features, encoding categoricals, and imputing remaining missing values. The result is saved to `../data/processed/train_processed.csv` for use in model training.

This mirrors the reusable pipeline in `src/preprocess.py`, broken out step by step for inspection.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Load Data

In [2]:
df = pd.read_csv("../data/raw/application_train.csv")
print(f"Loaded shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()


Loaded shape: 307,511 rows x 122 columns


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Drop Columns with >50% Missing Values

In [3]:
missing_pct = df.isnull().mean()
threshold = 0.5
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns with >{threshold*100:.0f}% missing values")
df = df.drop(columns=cols_to_drop)
print(f"Shape after drop: {df.shape}")


Dropping 41 columns with >50% missing values
Shape after drop: (307511, 81)


## 4. Transform DAYS Columns

Home Credit stores several time fields as negative day counts relative to the application date. Converting them to positive, human-readable years makes them far more interpretable for both modeling and explainability.

In [4]:
# DAYS_BIRTH -> AGE_YEARS
if "DAYS_BIRTH" in df.columns:
    df["AGE_YEARS"] = (-df["DAYS_BIRTH"]) / 365
    df = df.drop(columns=["DAYS_BIRTH"])

# DAYS_EMPLOYED -> YEARS_EMPLOYED, with an anomaly flag
# (Home Credit uses 365243 as a sentinel for "not employed" / pensioners)
if "DAYS_EMPLOYED" in df.columns:
    df["EMPLOYED_ANOMALY"] = (df["DAYS_EMPLOYED"] > 0).astype(int)
    df["YEARS_EMPLOYED"] = (-df["DAYS_EMPLOYED"].clip(upper=0)) / 365
    df = df.drop(columns=["DAYS_EMPLOYED"])

# Remaining DAYS_* columns -> YEARS_*
for col in ["DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE"]:
    if col in df.columns:
        df[col.replace("DAYS_", "YEARS_")] = (-df[col]) / 365
        df = df.drop(columns=[col])

print("DAYS_* columns transformed. New columns:",
      [c for c in df.columns if c.startswith(("AGE_", "YEARS_", "EMPLOYED_"))])


DAYS_* columns transformed. New columns: ['YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BEGINEXPLUATATION_MEDI', 'AGE_YEARS', 'EMPLOYED_ANOMALY', 'YEARS_EMPLOYED', 'YEARS_ID_PUBLISH', 'YEARS_LAST_PHONE_CHANGE']


## 5. Create Derived Features

Ratio and aggregate features that are commonly predictive of default risk: affordability ratios, income-per-person, implied loan term, and summary statistics across the three `EXT_SOURCE_*` credit bureau scores.

In [5]:
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + 1)
df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / (df["AMT_INCOME_TOTAL"] + 1)
df["CREDIT_GOODS_RATIO"] = df["AMT_CREDIT"] / (df["AMT_GOODS_PRICE"] + 1)
df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / (df["CNT_FAM_MEMBERS"] + 1)
df["LOAN_TERM_MONTHS"] = df["AMT_CREDIT"] / (df["AMT_ANNUITY"] + 1)

ext_cols = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in df.columns]
if ext_cols:
    df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
    df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
    df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

derived_cols = ["ANNUITY_INCOME_RATIO", "CREDIT_INCOME_RATIO", "CREDIT_GOODS_RATIO",
                "INCOME_PER_PERSON", "LOAN_TERM_MONTHS", "EXT_SOURCE_MEAN",
                "EXT_SOURCE_MIN", "EXT_SOURCE_STD"]
print("Derived features created:", derived_cols)
df[derived_cols].describe()


Derived features created: ['ANNUITY_INCOME_RATIO', 'CREDIT_INCOME_RATIO', 'CREDIT_GOODS_RATIO', 'INCOME_PER_PERSON', 'LOAN_TERM_MONTHS', 'EXT_SOURCE_MEAN', 'EXT_SOURCE_MIN', 'EXT_SOURCE_STD']


,ANNUITY_INCOME_RATIO,CREDIT_INCOME_RATIO,CREDIT_GOODS_RATIO,INCOME_PER_PERSON,LOAN_TERM_MONTHS,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_STD
count,307499.000000,307511.000000,307233.000000,3.075090e+05,307499.000000,3.072810e+05,3.072810e+05,2.461160e+05
mean,0.180928,3.957537,1.122992,5.748225e+04,21.611295,5.113215e-01,4.297357e-01,1.440541e-01
std,0.094573,2.689696,0.124044,6.585166e+04,7.823599,1.556360e-01,1.897950e-01,1.100530e-01
min,0.000224,0.004808,0.150000,2.647059e+03,8.036227,8.173617e-08,8.173617e-08,3.538459e-07
25%,0.114781,2.018659,0.999999,3.375000e+04,15.614247,4.136671e-01,2.833866e-01,5.490721e-02
50%,0.162832,3.265042,1.118798,4.950000e+04,19.999259,5.315221e-01,4.494371e-01,1.189984e-01
75%,0.229064,5.159857,1.197998,6.750000e+04,27.099077,6.310096e-01,5.859647e-01,2.120033e-01
max,1.875892,84.733539,5.999867,2.925000e+07,45.301400,8.633634e-01,8.633634e-01,5.948997e-01


## 6. Encode Categorical Features

Binary categoricals (two unique values) are label-encoded; higher-cardinality categoricals are one-hot encoded.

In [6]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"{len(cat_cols)} categorical columns to encode: {cat_cols}")

binary_cols = [c for c in cat_cols if df[c].nunique() == 2]
multi_cols = [c for c in cat_cols if df[c].nunique() > 2]

print(f"Label encoding {len(binary_cols)} binary columns: {binary_cols}")
for col in binary_cols:
    df[col] = pd.factorize(df[col])[0]

print(f"One-hot encoding {len(multi_cols)} multi-class columns: {multi_cols}")
df = pd.get_dummies(df, columns=multi_cols, drop_first=True, dtype=int)

print(f"Shape after encoding: {df.shape}")


13 categorical columns to encode: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'EMERGENCYSTATE_MODE']
Label encoding 4 binary columns: ['NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE']
One-hot encoding 9 multi-class columns: ['CODE_GENDER', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']
Shape after encoding: (307511, 190)


## 7. Median Impute Remaining Missing Values

In [7]:
numeric_cols = df.select_dtypes(include=np.number).columns
n_missing_before = df[numeric_cols].isnull().sum().sum()

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

n_missing_after = df[numeric_cols].isnull().sum().sum()
print(f"Missing values before imputation: {n_missing_before:,}")
print(f"Missing values after imputation: {n_missing_after:,}")


Missing values before imputation: 1,434,787
Missing values after imputation: 0


## 8. Final Shape and Target Distribution

In [8]:
print(f"Final processed shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print("\nTarget distribution:")
print(df["TARGET"].value_counts())
print(f"\nDefault rate: {df['TARGET'].mean() * 100:.2f}%")


Final processed shape: 307,511 rows x 190 columns

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Default rate: 8.07%


## 9. Save Processed Data

In [9]:
output_path = f"{OUTPUT_DIR}/train_processed.csv"
df.to_csv(output_path, index=False)
print(f"Saved processed dataset to {output_path}")
print(f"Final shape: {df.shape}")


Saved processed dataset to ../data/processed/train_processed.csv
Final shape: (307511, 190)
